# Chapter 08 — SOLUTIONS: Clustering Techniques

**Instructor file.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

sns.set_theme(style='whitegrid')
np.random.seed(42)

iris = load_iris()
X = iris.data
y_true = iris.target
X_scaled = StandardScaler().fit_transform(X)
print('Setup complete!')

## Task 1 Solution: Elbow Method

In [ ]:
k_values = range(1, 9)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_ for k in k_values]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_values), inertias, 'o-', color='#3498db', linewidth=2)
ax.axvline(3, color='#e74c3c', linestyle='--', label='Elbow at k=3')
ax.set_xlabel('k')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method — Iris Dataset')
ax.legend()
plt.tight_layout()
plt.show()
print('Elbow at k=3 — matches the 3 iris species!')

## Task 2 Solution: Silhouette Score

In [ ]:
print(f'{"k":>4}  {"Silhouette":>12}')
print('-' * 18)
best_k, best_sil = 2, -1
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    if sil > best_sil:
        best_k, best_sil = k, sil
    print(f'{k:>4}  {sil:>12.4f}')
print(f'\nBest k = {best_k} (silhouette = {best_sil:.4f})')

## Task 3 Solution: Cluster and Visualize

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', alpha=0.7, s=50)
# Centroids (in original space for plotting)
centers_orig = StandardScaler().fit(X).inverse_transform(kmeans.cluster_centers_)
ax.scatter(centers_orig[:, 0], centers_orig[:, 1], c='black', marker='X', s=200, zorder=10, label='Centroids')
ax.set_xlabel(iris.feature_names[0])
ax.set_ylabel(iris.feature_names[1])
ax.set_title('K-Means (k=3) on Iris Dataset')
ax.legend()
plt.tight_layout()
plt.show()

## Task 4 Solution: How Good Were We?

In [ ]:
ari = adjusted_rand_score(y_true, labels)
print(f'Adjusted Rand Index: {ari:.4f}')
print('(1.0 = perfect match, 0 = random, can be negative for worse than random)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (title, colors) in zip(axes, [
    ('K-Means clusters (our result)', labels),
    ('True species (hidden during clustering)', y_true)
]):
    ax.scatter(X[:, 0], X[:, 1], c=colors, cmap='tab10', alpha=0.7, s=50)
    ax.set_xlabel(iris.feature_names[0])
    ax.set_ylabel(iris.feature_names[1])
    ax.set_title(title, fontsize=12)

plt.suptitle(f'K-Means vs True Labels (ARI = {ari:.3f})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('Note: cluster colors may differ from species colors — label numbering is arbitrary!')

## Bonus Solution: DBSCAN

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()
ari_db     = adjusted_rand_score(y_true, db_labels)

print(f'DBSCAN found {n_clusters} clusters and {n_noise} noise points')
print(f'DBSCAN ARI: {ari_db:.4f}')
print(f'KMeans  ARI: {ari:.4f}')
print('\nOn this dataset, K-Means usually wins because clusters are roughly spherical.')
print('DBSCAN is harder to tune but great for irregular shapes and outlier detection.')

## Bonus Solution: Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# 1. Compute linkage and plot dendrogram
Z = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=20, leaf_rotation=45, leaf_font_size=10)
ax.axhline(y=6, color='red', linestyle='--', label='Cut at height=6 → 3 clusters')
ax.set_title('Hierarchical Clustering Dendrogram (Iris Dataset)', fontsize=13)
ax.set_xlabel('Sample index (or cluster size)')
ax.set_ylabel('Distance (Ward)')
ax.legend()
plt.tight_layout()
plt.show()
print('The dendrogram shows 3 natural clusters — matching the 3 Iris species!')

# 2. Cut at 3 clusters
hc_labels = fcluster(Z, t=3, criterion='maxclust') - 1  # 0-indexed

# 3. Compare ARI
ari_hc = adjusted_rand_score(y_true, hc_labels)
print(f'\nHierarchical Clustering ARI: {ari_hc:.4f}')
print(f'K-Means              ARI: {ari:.4f}')
print('\nBoth algorithms recover the Iris species structure well.')
print('Hierarchical clustering has the added benefit of showing the full merge tree,')